# EdgeGuard · Drive veri ön-hazırlığı

Bu notebook veri indirme yetkisi vermez ve lisans kabulünü otomatikleştirmez. Resmî paketleri Drive'a yerleştirdikten sonra klasör düzenini denetler ve her veri setini tek, SHA-256 bağlı `.tar` dosyasına dönüştürür. Eğitim notebook'u Drive'daki binlerce küçük dosyayı okumak yerine bu paketleri `/content` alanına taşır.

Çekirdek eğitim için yalnız **Cityscapes Fine + BDD100K 10K Semantic + IDD20K Part I/II** gerekir. ACDC ve kapalı external setler model dondurulmadan indirilmez.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPOSITORY = "https://github.com/emrealmaoglu/edgeguard-road.git"
BRANCH = "rescue/semantic-first"
PROJECT_ROOT = Path("/content/edgeguard-road")
DRIVE_ROOT = Path("/content/drive/MyDrive")
ACTIVE_SOURCE_DATASETS = ["cityscapes", "bdd100k", "idd20k"]
OPTIONAL_FINAL_DATASETS = []  # Model/protokol freeze sonrası ör. ["acdc"]
DATASETS_TO_BUNDLE = ACTIVE_SOURCE_DATASETS + OPTIONAL_FINAL_DATASETS
VERIFY_ARCHIVE_HASHES = True  # Resmî arşivleri bir kez SHA-256/MD5 ile kaydeder.
CREATE_BUNDLES = False  # Hazır klasörler denetlendikten sonra True yapın.
REPLACE_BUNDLES = False  # Yalnız kaynak klasörü bilinçli değiştiyse True yapın.

In [ ]:
import subprocess
import sys

if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPOSITORY, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)], check=True)

In [ ]:
# Drive klasörlerini oluştur, erişim talimatlarını ve eksikleri tek raporda göster.
import json

PREFLIGHT_REPORT = DRIVE_ROOT / "EdgeGuard/manifests/colab-data-inventory.json"
inventory_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/prepare_colab_data.py"),
    "--drive-root",
    str(DRIVE_ROOT),
    "--output",
    str(PREFLIGHT_REPORT),
    "inventory",
]
if VERIFY_ARCHIVE_HASHES:
    inventory_command.append("--hash-archives")
subprocess.run(inventory_command, check=True)
inventory = json.loads(PREFLIGHT_REPORT.read_text())
for row in inventory["datasets"]:
    print("\n", row["dataset_id"], "=>", row["state"], "|", row["activation_phase"])
    print("resmî kaynak:", row["official_url"])
    print("işlem:", row["instructions"])
    if row["missing_required_paths"]:
        print("eksik hazır yollar:", row["missing_required_paths"])
    for package in row["packages"]:
        print("paket:", package["filename"], "Drive'da:", package["present"])

## Manuel hazırlama hedefi

Resmî paketleri `MyDrive/EdgeGuard/archives/<dataset_id>/` altında saklayın; giriş bilgisi, cookie veya geçici indirme URL'sini notebook'a yazmayın. Paketleri açtıktan sonra aşağıdaki hazır kökleri oluşturun:

- `MyDrive/EdgeGuard/datasets/cityscapes/{leftImg8bit,gtFine}`
- `MyDrive/EdgeGuard/datasets/bdd100k/{images/10k,labels/sem_seg/masks}`
- `MyDrive/EdgeGuard/datasets/idd20k/{leftImg8bit,gtFine}` — Part I ve Part II aynı köke açılmalı.

Notebook yalnız klasörlerin varlığını değil, sonraki bilimsel audit'in beklediği train/val alt yollarını da kontrol eder. Arşiv adlarını değiştirmeyin; BDD paketlerinde yayımlanmış MD5 değerleri erişim planında kayıtlıdır. Bilinmeyen veya üçüncü taraf ayna kullanmayın.

In [ ]:
# Hazır kökler geçtikten sonra tek dosyalı, hash-bağlı Drive paketlerini üret.
if CREATE_BUNDLES:
    command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts/prepare_colab_data.py"),
        "--drive-root",
        str(DRIVE_ROOT),
        "bundle",
    ]
    for dataset in DATASETS_TO_BUNDLE:
        command.extend(["--dataset", dataset])
    if REPLACE_BUNDLES:
        command.append("--replace")
    subprocess.run(command, check=True)
else:
    print("CREATE_BUNDLES=False: manuel indirme/açma ve inventory incelemesi bekleniyor.")

In [ ]:
# Eğitim notebook'una geçiş kapısı: üç receipt ve üç tar dosyası birlikte bulunmalı.
bundle_root = DRIVE_ROOT / "EdgeGuard/bundles"
missing = []
for dataset in ACTIVE_SOURCE_DATASETS:
    for suffix in (".prepared.tar", ".prepared.tar.receipt.json"):
        candidate = bundle_root / f"{dataset}{suffix}"
        if not candidate.is_file():
            missing.append(str(candidate))
if missing:
    print("Eğitim öncesi eksikler:\n- " + "\n- ".join(missing))
else:
    print("VERİ HAZIRLIK KAPISI GEÇTİ — EdgeGuard_Road_Colab.ipynb açılabilir.")